In [ ]:

!pip uninstall openai-whisper numba -y -q
!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --break-system-packages -q
!pip install chatterbox-tts --break-system-packages -q
!pip install 'numpy>=1.24,<2.0' --force-reinstall --break-system-packages -q
!pip install faster-whisper --break-system-packages -q          # ← remplace openai-whisper
!pip install transformers sentence-transformers --break-system-packages -q
!pip install flask pyngrok --break-system-packages -q
print("✅ Toutes les dépendances installées")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 56.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-contrib-python 4.13.0.92 requires numpy>=2; py

In [ ]:
# ── CELL 1: Trouver la bonne version ─────────────────────────────────────────
!pip install 'transformers==4.46.0' --break-system-packages -q
print("✅ Redémarre le runtime (Runtime → Restart session)")

Reason for being yanked: This version unfortunately does not work with 3.8 but we did not drop the support yet
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
chatterbox-tts 0.1.7 requires transformers==5.2.0, but you have transformers 4.46.0 which is incompatible.
✅ Redémarre le runtime (Runtime → Restart session)


## Cellule 2 — Chargement de tous les modèles (une seule fois)

In [ ]:
import torch

from faster_whisper import WhisperModel
from transformers import pipeline as hf_pipeline
from sentence_transformers import SentenceTransformer
import numpy as np
import re
import torchaudio as ta
from chatterbox.tts import ChatterboxTTS

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'📍 Device : {DEVICE}')

print('🔊 Chargement Whisper...')
whisper_model = WhisperModel('base', device=DEVICE, compute_type='int8')
print('  ✓ Whisper prêt')

print('📝 Chargement BART...')

bart_pipe = hf_pipeline(
    'text2text-generation',
    model='facebook/bart-large-cnn',
    device=0 if DEVICE == 'cuda' else -1,
)
print('  ✓ BART prêt')

print('🔗 Chargement SentenceTransformer...')
sim_model = SentenceTransformer('all-MiniLM-L6-v2')
print('  ✓ SentenceTransformer prêt')

print('🗣️  Chargement Chatterbox TTS...')
tts_model = ChatterboxTTS.from_pretrained(device=DEVICE)
print('  ✓ Chatterbox TTS prêt')

print('\n🚀 Tous les modèles sont chargés !')

📍 Device : cuda
🔊 Chargement Whisper...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json: 0.00B [00:00, ?B/s]

vocabulary.txt: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

s3gen.safetensors:   0%|          | 0.00/1.06G [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

conds.pt:   0%|          | 0.00/107k [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/diffusers/models/lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)


loaded PerthNet (Implicit) at step 250,000
  ✓ Chatterbox TTS prêt

🚀 Tous les modèles sont chargés !


## ⚙️ Cellule 3 — Fonctions du pipeline

In [ ]:
import io, base64, tempfile, os


# ═══════════════════════════════════════════════════════════════
# CHUNKS — extraction et tri depuis la requête
# ═══════════════════════════════════════════════════════════════

def extract_ordered_chunk_keys(request_files) -> list:
    """
    Récupère les clés 'chunk_0', 'chunk_1', ... triées par index.
    Lève ValueError si aucun chunk trouvé.
    """
    keys = [k for k in request_files.keys() if k.startswith('chunk_')]
    if not keys:
        raise ValueError(
            "Aucun chunk trouvé. Envoie les fichiers audio avec les clés "
            "'chunk_0', 'chunk_1', 'chunk_2', ..."
        )
    keys.sort(key=lambda k: int(k.split('_')[1]))
    return keys


def save_chunk_tmp(file_obj) -> str:
    """Sauvegarde un fichier uploadé dans /tmp et retourne son chemin."""
    suffix = os.path.splitext(file_obj.filename)[-1] or '.wav'
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix=suffix)
    file_obj.save(tmp.name)
    return tmp.name


# ═══════════════════════════════════════════════════════════════
# TRANSCRIPTION + FUSION
# ═══════════════════════════════════════════════════════════════

def transcribe_all_chunks(request_files) -> str:
    """
    1. Récupère les chunks dans l'ordre (chunk_0, chunk_1, ...)
    2. Transcrit chaque chunk avec Whisper
    3. Fusionne les textes en un seul texte complet
    Retourne le texte complet fusionné.
    """
    keys = extract_ordered_chunk_keys(request_files)
    print(f'  📦 {len(keys)} chunks reçus : {keys}')

    transcriptions = []
    tmp_paths = []

    try:
        for i, key in enumerate(keys):
            print(f'  🎙️  Transcription {key} ({i+1}/{len(keys)})...')
            tmp_path = save_chunk_tmp(request_files[key])
            tmp_paths.append(tmp_path)


            file_size = os.path.getsize(tmp_path)
            print(f'      📁 Fichier : {tmp_path} ({file_size} bytes)')
            print(f'      🔧 whisper_model type : {type(whisper_model)}')

            try:
                segments, info = whisper_model.transcribe(tmp_path, beam_size=5)
                print(f'      🌍 Langue détectée : {info.language} ({info.language_probability:.2f})')

                seg_list = list(segments)
                print(f'      📝 {len(seg_list)} segments trouvés')

                text = ' '.join(seg.text for seg in seg_list).strip()
                print(f'      ✓ {len(text.split())} mots : "{text[:60]}..."')
            except Exception as e:
                import traceback
                print(f'      ✗ ERREUR transcription {key}:')
                traceback.print_exc()
                raise

            transcriptions.append(text)
    finally:
        for p in tmp_paths:
            if os.path.exists(p):
                os.unlink(p)

    full_text = ' '.join(transcriptions)
    print(f'  ✅ Fusion terminée : {len(full_text.split())} mots au total')
    return full_text


# ═══════════════════════════════════════════════════════════════
# RÉSUMÉ
# ═══════════════════════════════════════════════════════════════

def clean_text(text: str) -> str:
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', ' ', text)
    text = re.sub(r'[\ufffd\ufffe\uffff]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def deduplicate_sentences(text: str, min_length: int = 20) -> str:
    seen, result = set(), []
    for s in text.split('. '):
        s = s.strip()
        if s and s not in seen and len(s) > min_length:
            seen.add(s)
            result.append(s)
    return '. '.join(result)


def semantic_chunk_text(
    text: str,
    threshold: float = 0.55,
    min_words: int = 60,
    max_words: int = 450,
) -> list:
    """Découpe le texte en chunks sémantiquement cohérents."""
    sentences = [s.strip() for s in text.replace('\n', ' ').split('.') if s.strip()]
    if len(sentences) <= 1:
        return [text]

    embeddings = sim_model.encode(sentences, batch_size=32, show_progress_bar=False)
    chunks, current, current_words = [], [sentences[0]], len(sentences[0].split())

    for i in range(1, len(sentences)):
        w   = len(sentences[i].split())
        sim = float(
            np.dot(embeddings[i], embeddings[i-1]) /
            (np.linalg.norm(embeddings[i]) * np.linalg.norm(embeddings[i-1]) + 1e-8)
        )
        if sim < threshold or (current_words + w) > max_words:
            chunk = '. '.join(current) + '.'
            if current_words < min_words and chunks:
                chunks[-1] += ' ' + chunk
            else:
                chunks.append(chunk)
            current, current_words = [sentences[i]], w
        else:
            current.append(sentences[i])
            current_words += w

    if current:
        last = '. '.join(current) + '.'
        if current_words < min_words and chunks:
            chunks[-1] += ' ' + last
        else:
            chunks.append(last)
    return chunks


def summarize_text(text: str, target_words: int = 150) -> str:
    """
    Pipeline : nettoyage → chunking sémantique → BART par chunk
    → fusion → BART final calibré sur target_words.

    Args:
        text         : texte complet à résumer
        target_words : nombre de mots cible pour le résumé final
    """
    text = clean_text(text)

    if len(text.split()) <= target_words:
        print('  ℹ️  Texte déjà assez court, pas de résumé nécessaire')
        return text

    chunks = semantic_chunk_text(text)
    print(f'  📦 {len(chunks)} chunks sémantiques')

    chunk_summaries = []
    for i, chunk in enumerate(chunks):
        word_count = len(chunk.split())
        if word_count < 30:
            continue
        max_len = max(40, min(word_count // 2, 150))
        min_len = max(20, max_len // 3)
        try:

            out = bart_pipe(
                chunk,
                max_new_tokens=max_len,
                min_length=min_len,
                num_beams=4,
                no_repeat_ngram_size=3,
                do_sample=False,
                truncation=True,
            )
            chunk_summaries.append(deduplicate_sentences(out[0]['generated_text']))
            print(f'  ✓ Chunk {i+1}/{len(chunks)} résumé')
        except Exception as e:
            print(f'  ⚠️  Chunk {i+1} ignoré : {e}')

    if not chunk_summaries:
        return text[:500]

    merged = deduplicate_sentences(' '.join(chunk_summaries))

    final_max = max(60, min(target_words + 50, 400))
    final_min = max(20, target_words // 2)
    print(f'  🎯 Résumé final → cible {target_words} mots (BART min={final_min}, max={final_max})')


    final = bart_pipe(
        merged,
        max_new_tokens=final_max,
        min_length=final_min,
        num_beams=4,
        no_repeat_ngram_size=3,
        do_sample=False,
        truncation=True,
    )[0]['generated_text']

    return deduplicate_sentences(final)


# ═══════════════════════════════════════════════════════════════
# TTS
# ═══════════════════════════════════════════════════════════════

def generate_tts_bytes(
    text: str,
    ref_audio_path: str = None,
    exaggeration: float = 0.5,
    cfg_weight: float = 0.5,
) -> bytes:
    """Génère l'audio TTS du texte et retourne les bytes WAV."""
    wav = tts_model.generate(
        text,
        audio_prompt_path=ref_audio_path,
        exaggeration=exaggeration,
        cfg_weight=cfg_weight,
    )
    buf = io.BytesIO()
    ta.save(buf, wav, tts_model.sr, format='wav')
    buf.seek(0)
    return buf.read()


print('✅ Fonctions pipeline prêtes')

✅ Fonctions pipeline prêtes


## 🌐 Cellule 4 — API Flask + ngrok

In [ ]:
from flask import Flask, request, jsonify
from pyngrok import ngrok
import threading, tempfile, os, base64, traceback

app = Flask(__name__)



@app.route('/transcript', methods=['POST'])
def route_transcript():
    try:
        print('\n[/transcript]')
        nb_chunks     = len([k for k in request.files if k.startswith('chunk_')])
        transcription = transcribe_all_chunks(request.files)
        return jsonify({
            'transcription': transcription,
            'nb_chunks'    : nb_chunks,
            'word_count'   : len(transcription.split()),
        })
    except ValueError as e:
        return jsonify({'error': str(e)}), 400
    except Exception:
        return jsonify({'error': traceback.format_exc()}), 500



@app.route('/resume', methods=['POST'])
def route_resume():
    try:
        print('\n[/resume]')
        target_words = int(request.form.get('target_words', 150))

        print('  [1/2] Transcription des chunks...')
        transcription = transcribe_all_chunks(request.files)

        print('  [2/2] Résumé...')
        resume = summarize_text(transcription, target_words=target_words)
        print(f'  ✓ Résumé : {len(resume.split())} mots')

        return jsonify({
            'transcription': transcription,
            'resume'       : resume,
            'target_words' : target_words,
            'word_count'   : len(resume.split()),
        })
    except ValueError as e:
        return jsonify({'error': str(e)}), 400
    except Exception:
        return jsonify({'error': traceback.format_exc()}), 500



@app.route('/resumeaudio', methods=['POST'])
def route_resume_audio():
    ref_path = None
    try:
        print('\n[/resumeaudio]')
        target_words = int(request.form.get('target_words', 150))
        exaggeration = float(request.form.get('exaggeration', 0.5))
        cfg_weight   = float(request.form.get('cfg_weight', 0.5))

        # Voix de référence (optionnel)
        if 'reference' in request.files:
            ref_tmp = tempfile.NamedTemporaryFile(delete=False, suffix='.wav')
            request.files['reference'].save(ref_tmp.name)
            ref_path = ref_tmp.name

        # 1. Transcription + fusion des chunks
        print('  [1/3] Transcription des chunks...')
        transcription = transcribe_all_chunks(request.files)

        # 2. Résumé
        print('  [2/3] Résumé...')
        resume = summarize_text(transcription, target_words=target_words)
        print(f'  ✓ Résumé : {len(resume.split())} mots')

        # 3. Génération audio du résumé
        print('  [3/3] Génération TTS...')
        audio_bytes = generate_tts_bytes(
            text=resume,
            ref_audio_path=ref_path,
            exaggeration=exaggeration,
            cfg_weight=cfg_weight,
        )
        print('  ✓ Audio généré')

        return jsonify({
            'transcription': transcription,
            'resume'       : resume,
            'audio_base64' : base64.b64encode(audio_bytes).decode('utf-8'),
            'audio_format' : 'wav',
            'target_words' : target_words,
            'word_count'   : len(resume.split()),
        })

    except ValueError as e:
        return jsonify({'error': str(e)}), 400
    except Exception:
        return jsonify({'error': traceback.format_exc()}), 500
    finally:
        if ref_path and os.path.exists(ref_path):
            os.unlink(ref_path)


# ─────────────────────────────────────────────────────────────
# Health check
# ─────────────────────────────────────────────────────────────
@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'ok', 'device': DEVICE})


# ─────────────────────────────────────────────────────────────
# Démarrage ngrok + Flask
# ─────────────────────────────────────────────────────────────
ngrok.kill()

NGROK_TOKEN = '2xRpM51RznsqGfGjqIvzuAnOfJJ_2zMNuiFy8ixNSugF94Wbd'
ngrok.set_auth_token(NGROK_TOKEN)

tunnel   = ngrok.connect(5000)
BASE_URL = tunnel.public_url

print('\n' + '═'*55)
print('🌐  API publique prête !')
print('═'*55)
print(f'  Health       →  GET  {BASE_URL}/health')
print(f'  Transcription→  POST {BASE_URL}/transcript')
print(f'  Résumé texte →  POST {BASE_URL}/resume')
print(f'  Résumé audio →  POST {BASE_URL}/resumeaudio')
print('═'*55)

threading.Thread(
    target=lambda: app.run(port=5000, use_reloader=False)
).start()


═══════════════════════════════════════════════════════
🌐  API publique prête !
═══════════════════════════════════════════════════════
  Health       →  GET  https://e2fd-34-186-30-14.ngrok-free.app/health
  Transcription→  POST https://e2fd-34-186-30-14.ngrok-free.app/transcript
  Résumé texte →  POST https://e2fd-34-186-30-14.ngrok-free.app/resume
  Résumé audio →  POST https://e2fd-34-186-30-14.ngrok-free.app/resumeaudio
═══════════════════════════════════════════════════════


## 📋 Exemple d'appel depuis ton code Python local

```python
import requests, base64

API_URL     = 'https://xxxx.ngrok-free.app'  # URL affichée au démarrage
chunk_paths = ['chunk_0.wav', 'chunk_1.wav', 'chunk_2.wav']

# Helper : construire les fichiers dans l'ordre
def open_chunks(paths):
    return {
        f'chunk_{i}': (f'chunk_{i}.wav', open(p, 'rb'), 'audio/wav')
        for i, p in enumerate(paths)
    }

# ── 1. Transcription seulement ──────────────────────────────
r = requests.post(f'{API_URL}/transcript', files=open_chunks(chunk_paths))
print(r.json()['transcription'])

# ── 2. Résumé texte avec target_words ───────────────────────
r = requests.post(
    f'{API_URL}/resume',
    files=open_chunks(chunk_paths),
    data={'target_words': 200},
)
print(r.json()['resume'])

# ── 3. Pipeline complet — résumé + audio ────────────────────
files = open_chunks(chunk_paths)
files['reference'] = ('ref.wav', open('ma_voix.wav', 'rb'), 'audio/wav')  # optionnel

r = requests.post(
    f'{API_URL}/resumeaudio',
    files=files,
    data={'target_words': 150, 'exaggeration': 0.5, 'cfg_weight': 0.5},
    timeout=300,
)
d = r.json()
print(d['transcription'])
print(d['resume'])

# Décoder et sauvegarder l'audio
with open('resume_audio.wav', 'wb') as f:
    f.write(base64.b64decode(d['audio_base64']))
```